# 01 — Series y DataFrames

Las dos estructuras de datos centrales de pandas. Todo lo que se hace en pandas es construir, transformar, o combinar Series y DataFrames.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
train = pd.read_csv(TRAIN, low_memory=False)
print(type(train))
print(train.shape)


<class 'pandas.DataFrame'>
(9800, 18)


## Series

Una Serie es un array unidimensional con un índice. Cada columna de un DataFrame es una Serie.

In [2]:
# Una columna extraída del DataFrame es una Series
ventas = train['Sales']
print(type(ventas))
print(ventas.dtype)
print(ventas.shape)   # (n,) — una dimensión
print()

# El índice por defecto es un RangeIndex (0, 1, 2, ...)
print(ventas.index)
print(ventas.head())


<class 'pandas.Series'>
float64
(9800,)

RangeIndex(start=0, stop=9800, step=1)
0    261.9600
1    731.9400
2     14.6200
3    957.5775
4     22.3680
Name: Sales, dtype: float64


In [3]:
# Crear una Series desde un diccionario — las claves se convierten en índice
ventas_region = pd.Series({
    'West':  827000,
    'East':  678000,
    'South': 391000,
    'Central': 501000,
})
print(ventas_region)
print()

# Operaciones vectorizadas — se aplican a todos los elementos sin loop
print(ventas_region * 1.21)        # con IVA
print(ventas_region[ventas_region > 500000])  # filtro


West       827000
East       678000
South      391000
Central    501000
dtype: int64

West       1000670.0
East        820380.0
South       473110.0
Central     606210.0
dtype: float64
West       827000
East       678000
Central    501000
dtype: int64


## DataFrame

Un DataFrame es una colección de Series que comparten el mismo índice. Es la estructura principal de trabajo.

In [4]:
# Crear desde diccionario de listas — las claves son nombres de columna
df_manual = pd.DataFrame({
    'producto': ['Laptop', 'Mouse', 'Monitor'],
    'precio':   [899.99, 15.50, 350.00],
    'stock':    [12, 200, 45],
})
print(df_manual)
print()
print('Tipos:', df_manual.dtypes.to_dict())


  producto  precio  stock
0   Laptop  899.99     12
1    Mouse   15.50    200
2  Monitor  350.00     45

Tipos: {'producto': <StringDtype(storage='python', na_value=nan)>, 'precio': dtype('float64'), 'stock': dtype('int64')}


## Atributos fundamentales

In [5]:
print('shape:  ', train.shape)            # (filas, columnas)
print('columns:', train.columns.tolist())
print('dtypes:')
print(train.dtypes)
print()
print('index:  ', train.index)             # RangeIndex por defecto
print('ndim:   ', train.ndim)              # siempre 2 para DataFrame
print('size:   ', train.size)              # total de celdas (filas × columnas)


shape:   (9800, 18)
columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']
dtypes:
Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code      float64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
dtype: object

index:   RangeIndex(start=0, stop=9800, step=1)
ndim:    2
size:    176400


## iloc vs loc

`iloc` selecciona por posición numérica (como un array). `loc` selecciona por etiqueta del índice. Con el índice por defecto (0, 1, 2…) ambos dan el mismo resultado, pero cuando el índice tiene etiquetas distintas, se comportan diferente.

In [6]:
# iloc — posición: fila 0, columnas 0 a 3
print(train.iloc[0, :4])
print()

# loc — etiqueta: fila con índice 0, columnas por nombre
print(train.loc[0, ['Order ID', 'Customer Name', 'Sales']])
print()

# La diferencia importa cuando el índice no empieza en 0
df_filtrado = train[train['Region'] == 'West'].copy()
print('iloc[0]:     ', df_filtrado.iloc[0]['Customer Name'])  # primer elemento de la selección
# loc[0] aquí buscaría la fila con índice 0 — puede no existir en df_filtrado


Row ID                     1
Order ID      CA-2017-152156
Order Date        08/11/2017
Ship Date         11/11/2017
Name: 0, dtype: object

Order ID         CA-2017-152156
Customer Name       Claire Gute
Sales                    261.96
Name: 0, dtype: object

iloc[0]:      Darrin Van Huff


In [7]:
# Slicing con iloc — igual que Python: inicio:fin:paso
print(train.iloc[:5, [0, 2, 4, 17]])   # primeras 5 filas, columnas por posición
print()

# Slicing con loc — el fin SÍ se incluye (diferente a Python)
print(train.loc[:4, 'Order ID':'Region'])  # filas 0-4, columnas 'Order ID' hasta 'Region'


   Row ID  Order Date       Ship Mode     Sales
0       1  08/11/2017    Second Class  261.9600
1       2  08/11/2017    Second Class  731.9400
2       3  12/06/2017    Second Class   14.6200
3       4  11/10/2016  Standard Class  957.5775
4       5  11/10/2016  Standard Class   22.3680

         Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
1  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
2  CA-2017-138688  12/06/2017  16/06/2017    Second Class    DV-13045   
3  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   
4  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   

     Customer Name    Segment        Country             City       State  \
0      Claire Gute   Consumer  United States        Henderson    Kentucky   
1      Claire Gute   Consumer  United States        Henderson    Kentucky   
2  Darrin Van Huff  Corporate  United Sta

## El índice

El índice es la etiqueta de cada fila. Por defecto es un `RangeIndex` (0, 1, 2…), pero puede ser cualquier valor único. `set_index()` convierte una columna en índice; `reset_index()` la devuelve como columna.

In [8]:
# Convertir 'Order ID' en índice
df_idx = train.set_index('Order ID')
print(df_idx.index[:5])
print()

# Ahora loc usa el Order ID como etiqueta
print(df_idx.loc['CA-2017-152156'])
print()

# reset_index devuelve el índice como columna
df_vuelta = df_idx.reset_index()
print(df_vuelta.columns.tolist())


Index(['CA-2017-152156', 'CA-2017-152156', 'CA-2017-138688', 'US-2016-108966',
       'US-2016-108966'],
      dtype='str', name='Order ID')

                Row ID  Order Date   Ship Date     Ship Mode Customer ID  \
Order ID                                                                   
CA-2017-152156       1  08/11/2017  11/11/2017  Second Class    CG-12520   
CA-2017-152156       2  08/11/2017  11/11/2017  Second Class    CG-12520   

               Customer Name   Segment        Country       City     State  \
Order ID                                                                     
CA-2017-152156   Claire Gute  Consumer  United States  Henderson  Kentucky   
CA-2017-152156   Claire Gute  Consumer  United States  Henderson  Kentucky   

                Postal Code Region       Product ID   Category Sub-Category  \
Order ID                                                                      
CA-2017-152156      42420.0  South  FUR-BO-10001798  Furniture    Bookcases   
CA-

---
## Resumen

| Concepto | Descripción |
|----------|-------------|
| `Series` | Array 1D con índice. Cada columna de un DataFrame es una Series |
| `DataFrame` | Colección de Series con índice compartido |
| `.shape` | `(filas, columnas)` |
| `.dtypes` | Tipo de dato de cada columna |
| `iloc[f, c]` | Selección por posición numérica |
| `loc[f, c]` | Selección por etiqueta de índice |
| `set_index()` | Convierte una columna en índice |
| `reset_index()` | Devuelve el índice como columna |
